In [2]:
# ============================================================
# Cell 1: Setup
# ============================================================
import json
import pandas as pd
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]
MAX_FRET = 24

# Update these to match your Drive layout
JAMS_DIR        = Path('/content/drive/MyDrive/Capstone/GuitarSet/Annotations')
PREDICTIONS_CSV = Path('/content/drive/MyDrive/Capstone/outputs/fretboard_playability/fretboard_predictions_heldout_test.csv')

Mounted at /content/drive


In [3]:
# ============================================================
# Cell 2: JAMS parser (same as ground-truth tab notebook)
# ============================================================
# We only need this for beats and tempo — the actual note positions
# come from the predictions CSV. The beat grid keeps the rendered tab
# musically aligned rather than spaced by raw seconds.

def _annotation_rows(annotation):
    data = annotation.get('data', [])
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        keys = ['time', 'duration', 'value', 'confidence']
        n = len(data.get('time', []))
        return [{k: data.get(k, [None] * n)[i] for k in keys} for i in range(n)]
    return []


def parse_jams_for_metadata(jams_path):
    """Lightweight parser — just the beats and tempo this notebook needs.
    The full notes/chords data lives in the predictions CSV."""
    jams_path = Path(jams_path)
    if not jams_path.exists():
        return {'beats': [], 'tempo': None, 'key': None}

    with open(jams_path) as f:
        jam = json.load(f)

    beats = []
    tempo = None
    key = None

    for ann in jam.get('annotations', []):
        ns = ann.get('namespace', '')
        rows = _annotation_rows(ann)
        if ns in ('beat', 'beat_position'):
            for r in rows:
                beats.append(float(r.get('time', 0.0)))
        elif ns == 'tempo' and rows:
            tempo = float(rows[0].get('value', 0.0)) or None
        elif ns == 'key_mode' and rows:
            key = rows[0].get('value')

    # Fall back to filename-encoded tempo
    if tempo is None:
        try:
            tempo = float(jams_path.stem.split('_')[1].split('-')[1])
        except (IndexError, ValueError):
            pass

    beats.sort()
    return {'beats': beats, 'tempo': tempo, 'key': key}

In [4]:
# ============================================================
# Cell 3: ASCII tab renderer (same as ground-truth tab notebook)
# ============================================================
DISPLAY_TO_STRING = [5, 4, 3, 2, 1, 0]   # high E on top, low E on bottom
STRING_LABELS     = ['e', 'B', 'G', 'D', 'A', 'E']


def _build_time_grid(beats, tempo, end_time, subdivisions_per_beat):
    if beats and len(beats) >= 2:
        grid = []
        for i in range(len(beats) - 1):
            step = (beats[i + 1] - beats[i]) / subdivisions_per_beat
            for j in range(subdivisions_per_beat):
                grid.append(beats[i] + j * step)
        last_step = (beats[-1] - beats[-2]) / subdivisions_per_beat
        while grid[-1] < end_time:
            grid.append(grid[-1] + last_step)
        return grid
    if tempo and tempo > 0:
        step = 60.0 / tempo / subdivisions_per_beat
        n = int(end_time / step) + subdivisions_per_beat
        return [i * step for i in range(n)]
    return [i * 0.25 for i in range(int(end_time / 0.25) + 2)]


def render_ascii_tab(parsed, subdivisions_per_beat=2, beats_per_measure=4,
                     measures_per_line=4, col_width=3, max_notes=None):
    notes = parsed['notes']
    if max_notes is not None:
        notes = notes[:max_notes]
    if not notes:
        return '(no notes)'

    end_time = max(n['start'] + n.get('duration', 0.5) for n in notes) + 0.5
    grid = _build_time_grid(
        parsed.get('beats', []), parsed.get('tempo'),
        end_time, subdivisions_per_beat,
    )
    n_cols = len(grid)
    cells = [[None] * n_cols for _ in range(6)]
    collisions = 0

    for note in notes:
        col = min(range(n_cols), key=lambda i: abs(grid[i] - note['start']))
        try:
            row = DISPLAY_TO_STRING.index(note['string'])
        except ValueError:
            continue
        if cells[row][col] is not None:
            collisions += 1
        cells[row][col] = note['fret']

    def fmt(v):
        if v is None:
            return '-' * col_width
        s = str(v)
        return s[:col_width] if len(s) >= col_width else s + '-' * (col_width - len(s))

    formatted = [[fmt(cells[r][c]) for c in range(n_cols)] for r in range(6)]
    cols_per_measure = beats_per_measure * subdivisions_per_beat
    cols_per_line = cols_per_measure * measures_per_line

    lines = []
    for start in range(0, n_cols, cols_per_line):
        end = min(start + cols_per_line, n_cols)
        for row in range(6):
            parts = []
            for c in range(start, end):
                if c > start and (c - start) % cols_per_measure == 0:
                    parts.append('|')
                parts.append(formatted[row][c])
            lines.append(f"{STRING_LABELS[row]}|{''.join(parts)}|")
        lines.append('')

    out = '\n'.join(lines)
    if collisions:
        out += f"\n[note: {collisions} cell collisions]"
    return out

In [5]:
# ============================================================
# Cell 4: Predictions → render-ready dict
# ============================================================

def load_predictions(csv_path=PREDICTIONS_CSV):
    """Load the algorithm's saved predictions for all (recording, method)
    combinations. Each row has start, duration, midi, true_string,
    true_fret, pred_string, pred_fret, recording, method."""
    df = pd.read_csv(csv_path)
    return df


def list_options(predictions_df):
    """Show what's available in the predictions CSV — useful for picking
    a recording or method to render."""
    print(f"{predictions_df['recording'].nunique()} recordings, "
          f"{predictions_df['method'].nunique()} methods.\n")
    print("Methods:")
    for m in sorted(predictions_df['method'].unique()):
        print(f"  - {m}")
    print(f"\nFirst 10 recordings:")
    for r in sorted(predictions_df['recording'].unique())[:10]:
        print(f"  - {r}")


def build_render_dict(predictions_df, recording_id, method='combined_all_tuned',
                      source='pred', jams_dir=JAMS_DIR):
    """Build the dict that render_ascii_tab expects.

    Args:
        predictions_df: DataFrame from load_predictions()
        recording_id:   e.g. '03_Jazz2-110-Bb_comp'
        method:         which assigner's predictions to render
        source:         'pred' for algorithm output, 'truth' for GuitarSet GT.
                        Same JAMS metadata either way — only the (string, fret)
                        columns differ. This makes side-by-side comparison clean.
        jams_dir:       where to find the .jams file for beats/tempo

    Returns:
        dict suitable for render_ascii_tab().
    """
    if source not in ('pred', 'truth'):
        raise ValueError("source must be 'pred' or 'truth'")

    rec_df = predictions_df[
        (predictions_df['recording'] == recording_id) &
        (predictions_df['method'] == method)
    ].sort_values('start')
    if rec_df.empty:
        raise ValueError(f"No data for recording={recording_id}, method={method}")

    s_col = 'pred_string' if source == 'pred' else 'true_string'
    f_col = 'pred_fret'   if source == 'pred' else 'true_fret'

    notes = []
    for _, row in rec_df.iterrows():
        if pd.isna(row[s_col]) or pd.isna(row[f_col]):
            continue
        notes.append({
            'start':    float(row['start']),
            'duration': float(row['duration']) if 'duration' in row and pd.notna(row['duration']) else 0.5,
            'midi':     int(row['midi']),
            'string':   int(row[s_col]),
            'fret':     int(row[f_col]),
        })

    meta = parse_jams_for_metadata(jams_dir / f"{recording_id}.jams")
    return {
        'recording': recording_id,
        'notes':     notes,
        'beats':     meta['beats'],
        'tempo':     meta['tempo'],
        'key':       meta['key'],
    }


def recording_accuracy(predictions_df, recording_id, method):
    """Quick exact-match number for the selected recording, useful as a
    headline alongside the rendered tab."""
    rec_df = predictions_df[
        (predictions_df['recording'] == recording_id) &
        (predictions_df['method'] == method)
    ]
    if rec_df.empty:
        return None
    matches = ((rec_df['pred_string'] == rec_df['true_string']) &
               (rec_df['pred_fret'] == rec_df['true_fret']))
    return matches.mean()

In [6]:
# ============================================================
# Cell 5: Demo — algorithm output vs ground truth
# ============================================================

preds = load_predictions()
list_options(preds)

54 recordings, 8 methods.

Methods:
  - combined_all
  - combined_all_tuned
  - highest_string
  - lowest_fret
  - nearest_previous
  - old_music_theory_greedy
  - viterbi_original
  - viterbi_playability

First 10 recordings:
  - 00_Funk1-114-Ab_solo
  - 00_Funk1-97-C_solo
  - 00_Funk2-108-Eb_solo
  - 00_Jazz3-150-C_comp
  - 00_Rock2-142-D_comp
  - 00_Rock2-85-F_comp
  - 00_Rock3-117-Bb_solo
  - 00_Rock3-148-C_solo
  - 00_SS2-107-Ab_solo
  - 00_SS3-84-Bb_solo


In [13]:
# ============================================================
# Cell 6: Render a chosen recording
# ============================================================

# Pick anything from the list above. A few suggestions:
#   '03_Jazz2-110-Bb_comp'  → biggest combined_all_tuned win (85%)
#   '03_BN1-147-Gb_solo'    → biggest combined_all_tuned loss (28%)
#   '05_Rock1-90-C#_comp'   → cleanest validation case (95%+)
RECORDING = '00_Jazz3-150-C_comp'
METHOD    = 'combined_all_tuned'

acc = recording_accuracy(preds, RECORDING, METHOD)
print(f"Recording: {RECORDING}")
print(f"Method:    {METHOD}")
print(f"Accuracy:  {acc:.1%}\n")

print("=" * 60)
print("ALGORITHM OUTPUT")
print("=" * 60)
pred_dict = build_render_dict(preds, RECORDING, METHOD, source='pred')
print(render_ascii_tab(pred_dict, max_notes=120))

print("\n" + "=" * 60)
print("GROUND TRUTH")
print("=" * 60)
gt_dict = build_render_dict(preds, RECORDING, METHOD, source='truth')
print(render_ascii_tab(gt_dict, max_notes=120))

Recording: 00_Jazz3-150-C_comp
Method:    combined_all_tuned
Accuracy:  93.7%

ALGORITHM OUTPUT
e|12----12----12----10-10-|------10-9-----------8--|---8-----8-----7--7-----|------8--8--------8--8--|
B|12----12----12----------|12-------12-------5-----|---5-----5-----8--8-----|8-----10-10-------10-8--|
G|12----12----12----------|11-------11----5--5-----|---5-----5-----7--7-----|7-----9--9--------9--9--|
D|10----10----------12----|12----12-12----5--5-----|---5-----5-----9--------|9--10----10-------10-10-|
A|------------------------|10----10----7--7--------|---------------7--------|7--8-----8--------------|
E|------------------------|---------------5--------|------------------------|------------------------|

e|---8-----8--------8--8--|------8-----8-----10-10-|---10-------------|
B|---8-----8-----8--------|10----10-10-10----10-10-|------------------|
G|0-----5--9-----9--9-----|9-----9--9--9-----10-10-|------------------|
D|------------------10----|10----10-10-10----9--9--|9----------------

In [14]:
print(render_ascii_tab(pred_dict, max_notes=None))


e|12----12----12----10-10-|------10-9-----------8--|---8-----8-----7--7-----|------8--8--------8--8--|
B|12----12----12----------|12-------12-------5-----|---5-----5-----8--8-----|8-----10-10-------10-8--|
G|12----12----12----------|11-------11----5--5-----|---5-----5-----7--7-----|7-----9--9--------9--9--|
D|10----10----------12----|12----12-12----5--5-----|---5-----5-----9--------|9--10----10-------10-10-|
A|------------------------|10----10----7--7--------|---------------7--------|7--8-----8--------------|
E|------------------------|---------------5--------|------------------------|------------------------|

e|---8-----8--------8--8--|------8-----8-----10-10-|---10-10-------8--8-----|8-----8--------10-10-6--|
B|---8-----8-----8--------|10----10-10-10----10-10-|---10-10----------------|10-------------12-7-----|
G|0-----5--9-----9--9-----|9-----9--9--9-----10-10-|---10-10-10----9--9-----|9-----9--11----11-------|
D|------------------10----|10----10-10-10----9--9--|9--9--9--9-----10-10

In [15]:
print(render_ascii_tab(gt_dict, max_notes=None))


e|12----12----12----10-10-|------10-9-----------8--|---8-----8-----7--7-----|------8--8--------8--8--|
B|12----12----12----------|12-------12-------5-----|---5-----5-----8--8-----|8-----10-10-------10-8--|
G|12----12----12----------|11-------11----5--5-----|---5-----5-----7--7-----|7-----9--9--------9--9--|
D|10----10----------12----|12----12-12----5--5-----|---5-----5-----9--------|9--10----10-------10-10-|
A|------------------------|10----10----7--7--------|---------------7--------|7--8-----8--------------|
E|------------------------|---------------5--------|------------------------|------------------------|

e|---8-----8--------8--8--|------8-----8-----10-10-|---10-10-------8--8-----|8-----8--------10-10-6--|
B|---8-----8-----8-----10-|10----10-10-10----10-10-|---10-10----------------|10----10-------12-------|
G|---------9-----9--9-----|9-----9--9--9-----10-10-|---10-10-10----9--9-----|9-----9--11----11-11----|
D|------10----------10----|10----10-10-10----9--9--|9--9--9--9-----10-10